# Notebook 05 — Linear Regression: Geometric & Probabilistic Interpretations

**Difficulty:** ⭐⭐⭐ | **Estimated time:** 2.5 hours  
**Theme:** Predicting house sale prices  
**Week:** 5 — ML Fundamentals & Linear Regression

---

## Why Does This Matter?

Most introductions to linear regression say: *"draw a line through the points."* That is correct but deeply unsatisfying. It leaves open the questions:

- **Why** minimize squared errors and not absolute errors or cubed errors?
- **Why** does the solution involving matrix inverses actually make geometric sense?
- **When** is OLS the *right* tool, and when should you use something else?

This notebook provides two complete, rigorous answers — the **geometric** view and the **probabilistic** view — and shows that they are secretly the same answer.

Understanding both interpretations will help you:
- Debug models (e.g., why are my residuals right-skewed?)
- Choose better loss functions for non-Gaussian errors
- Understand regularization (ridge/lasso) as a principled statistical choice, not just a trick

## The Big Picture: Two Views, One Formula

| Interpretation | Central Question | Key Insight |
|---|---|---|
| **Geometric** | Where does ŷ live in space? | ŷ = projection of y onto column space of X |
| **Probabilistic** | What θ makes the data most likely? | MLE under Gaussian noise = minimizing MSE |

Both perspectives arrive at the same closed-form solution: **θ* = (XᵀX)⁻¹Xᵀy**

---

## Section 1 — Geometric Interpretation: Linear Regression as Projection

### Real-World Analogy First

Imagine you are standing in a room holding a ball. There is a flat wall in front of you. You shine a flashlight straight at the wall — the shadow of the ball on the wall is the **projection** of the ball onto the wall's surface.

- The **ball** = the true house prices vector **y** (lives in full n-dimensional space)
- The **wall** = the column space of X (all predictions our features can produce)
- The **shadow** = **ŷ = Xθ** (the closest we can get to y using only our features)
- The **flashlight beam** = the residual vector **(y − ŷ)** — it hits the wall at a right angle

The key insight: **the beam is perpendicular to the wall**. This is not a coincidence — it is the mathematical definition of the closest point.

### Plain English

Our feature matrix X can only produce a limited set of predictions (its column space). The true prices **y** almost certainly cannot be perfectly expressed by those features. OLS finds the point inside the "wall" (column space) that is **closest** to y. Closest means smallest total squared distance — which is exactly the MSE objective.

### The Mathematics

The column space of X is: **col(X) = {Xθ : θ ∈ ℝᵖ}**

The projection ŷ satisfies: **(y − ŷ) ⊥ col(X)**

Which means for every column xⱼ of X:  **xⱼᵀ(y − ŷ) = 0**

In matrix form: **Xᵀ(y − Xθ) = 0** → **XᵀXθ = Xᵀy** → **θ* = (XᵀX)⁻¹Xᵀy**

The matrix H = X(XᵀX)⁻¹Xᵀ is called the **hat matrix** because it puts the hat on y: **ŷ = Hy**

In [ ]:
# ── Imports and reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)                      # reproducible results throughout
plt.rcParams['figure.dpi'] = 110        # crisper plots
plt.rcParams['font.size'] = 11

print("All libraries imported successfully.")
print(f"NumPy {np.__version__}  |  Pandas {pd.__version__}")

In [ ]:
# ── Load California Housing — single-feature version ─────────────────────────
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

# Use median_income (MedInc) as our single predictor of house value
# This gives us a 2D problem we can actually visualise
df = df[['MedInc', 'MedHouseVal']].dropna()

# Sub-sample for clarity in plots (full dataset has 20,640 rows)
df_sample = df.sample(n=300, random_state=42).reset_index(drop=True)

X_raw = df_sample[['MedInc']].values          # shape (300, 1)
y     = df_sample['MedHouseVal'].values        # shape (300,)

# Add bias column manually: X = [1, MedInc] so θ = [intercept, slope]
n = len(y)
X = np.column_stack([np.ones(n), X_raw])      # shape (300, 2)

print(f"Dataset shape  — X: {X.shape}  |  y: {y.shape}")
print(f"Feature range  — MedInc: [{X_raw.min():.2f}, {X_raw.max():.2f}]")
print(f"Target range   — MedHouseVal: [{y.min():.2f}, {y.max():.2f}] (in $100k units)")
df_sample.describe().round(3)

In [ ]:
# ── OLS solution via normal equations ────────────────────────────────────────
# θ* = (XᵀX)⁻¹ Xᵀy  — the closed-form solution we will derive in Notebook 06
XtX   = X.T @ X                         # (p × p) matrix  p=2 here
Xty   = X.T @ y                         # (p,) vector
theta = np.linalg.inv(XtX) @ Xty        # solve the normal equations exactly

y_hat     = X @ theta                   # fitted values (the projection)
residuals = y - y_hat                   # residuals = y - ŷ

print("OLS Estimates:")
print(f"  Intercept  θ₀ = {theta[0]:.4f}  (base house price when income = 0)")
print(f"  Slope      θ₁ = {theta[1]:.4f}  ($100k increase per unit of MedInc)")
print(f"  Residual mean  = {residuals.mean():.6f}  (should be ≈ 0)")
print(f"  RSS            = {(residuals**2).sum():.4f}")

### Verifying Orthogonality: The Core of Geometric Interpretation

The geometric interpretation says: **Xᵀ(y − ŷ) = 0** — the residuals are orthogonal to every column of X.

Let's verify this numerically. If it holds, then ŷ truly is the closest point in col(X) to y.

In [ ]:
# ── Orthogonality check: Xᵀ · residuals ≈ 0 ─────────────────────────────────
orthogonality_check = X.T @ residuals

print("Xᵀ · (y - ŷ) =", orthogonality_check)
print()

# np.allclose checks element-wise closeness with tolerance atol
is_orthogonal = np.allclose(X.T @ residuals, 0, atol=1e-6)
print(f"Are residuals orthogonal to X?  →  {is_orthogonal}")
print()

# Breaking it down per column of X:
#   Column 0 of X is the ones vector — Xᵀ₀·residuals = sum of residuals ≈ 0
#   Column 1 of X is MedInc — the weighted sum of (MedInc × residuals) ≈ 0
print("Interpretation:")
print(f"  Σ residuals                = {orthogonality_check[0]:.2e}  (≈ 0 means zero mean)")
print(f"  Σ MedInc × residuals       = {orthogonality_check[1]:.2e}  (≈ 0 means decorrelated)")
print()
print("CONCLUSION: OLS residuals are perpendicular to every feature — this is")
print("the algebraic proof that ŷ is the projection of y onto col(X).")

In [ ]:
# ── Geometric visualisation: y, ŷ, and residuals ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left panel: scatter with regression line and residuals ───────────────────
ax = axes[0]
ax.scatter(X_raw.ravel(), y, alpha=0.45, color='steelblue', s=25, label='Observed y')

# Draw the regression line across the data range
x_line = np.linspace(X_raw.min(), X_raw.max(), 200)
y_line = theta[0] + theta[1] * x_line
ax.plot(x_line, y_line, color='crimson', linewidth=2, label='ŷ = Xθ (projection)')

# Draw a sample of residuals as vertical arrows (perpendicular to x-axis in 2D)
idx_sample = np.random.choice(n, 30, replace=False)   # pick 30 random points
for i in idx_sample:
    ax.annotate('', xy=(X_raw[i, 0], y_hat[i]),
                xytext=(X_raw[i, 0], y[i]),
                arrowprops=dict(arrowstyle='->', color='orange', lw=1.2))

ax.set_xlabel('Median Income (MedInc)')
ax.set_ylabel('Median House Value ($100k)')
ax.set_title('Projection: y → col(X)\n(Arrows = residuals y − ŷ)', fontsize=12)
ax.legend()

# ── Right panel: residual distribution (should cluster around 0) ─────────────
ax2 = axes[1]
ax2.hist(residuals, bins=30, color='steelblue', alpha=0.7, edgecolor='white', density=True)
xr = np.linspace(residuals.min(), residuals.max(), 200)
ax2.plot(xr, stats.norm.pdf(xr, residuals.mean(), residuals.std()),
         color='crimson', linewidth=2, label='N(0, σ²) fit')
ax2.axvline(0, color='black', linestyle='--', linewidth=1)
ax2.set_xlabel('Residual (y − ŷ)')
ax2.set_ylabel('Density')
ax2.set_title('Residual Distribution\n(Is Gaussian assumption reasonable?)', fontsize=12)
ax2.legend()

plt.tight_layout()
plt.savefig('geo_projection.png', bbox_inches='tight', dpi=110)
plt.show()
print("Left: ŷ is the projection of y onto col(X). Arrows show residuals y − ŷ.")
print("Right: If the distribution looks roughly bell-shaped, Gaussian noise is plausible.")

## Section 2 — Probabilistic Interpretation: Maximum Likelihood Estimation

### Real-World Analogy First

You are a detective. You have a suspect (θ) and evidence (y). You ask: **"Given this suspect, how likely would I be to see this exact set of evidence?"** You try many suspects and pick the one that makes the evidence *most likely*. That is Maximum Likelihood Estimation.

In house pricing: given a model θ, how likely is it that we would observe *exactly* these house prices? The answer depends on how far the actual prices are from our predictions — the residuals. If residuals are small, the data is very likely; if large, less likely.

### The Probabilistic Model

We assume:

$$y_i = x_i^\top \theta + \varepsilon_i \quad \text{where} \quad \varepsilon_i \sim \mathcal{N}(0, \sigma^2)$$

This means each house price is our prediction plus a small Gaussian noise term. Therefore:

$$P(y_i \mid x_i, \theta) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{(y_i - x_i^\top\theta)^2}{2\sigma^2}\right)$$

### The Log-Likelihood

For all n observations (assuming independence):

$$\log \mathcal{L}(\theta) = -\frac{n}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^n (y_i - x_i^\top\theta)^2$$

### The Key Insight

To **maximise** log L(θ), we need to **minimise** $\sum_i (y_i - x_i^\top\theta)^2$ — which is exactly the MSE objective.

**OLS = MLE under Gaussian noise. They are the same optimisation problem.**

In [ ]:
# ── Log-likelihood landscape: visualise as a function of one parameter ────────
# Fix intercept at OLS value; vary only the slope θ₁ to make a 1D plot

# Estimate σ² from residuals (MLE estimate of noise variance)
sigma2_hat = np.var(residuals, ddof=0)   # biased MLE estimator: 1/n * RSS

def log_likelihood(theta1_vals, theta0_fixed, X_raw, y, sigma2):
    """Compute log-likelihood for a range of slope values, fixed intercept."""
    ll = []
    n  = len(y)
    for t1 in theta1_vals:
        y_pred = theta0_fixed + t1 * X_raw.ravel()  # prediction at this theta
        rss    = np.sum((y - y_pred)**2)             # residual sum of squares
        # log L = -n/2 * log(2π σ²) - RSS / (2σ²)
        ll_val = -n/2 * np.log(2 * np.pi * sigma2) - rss / (2 * sigma2)
        ll.append(ll_val)
    return np.array(ll)

theta1_grid = np.linspace(theta[1] - 0.6, theta[1] + 0.6, 400)
ll_values   = log_likelihood(theta1_grid, theta[0], X_raw, y, sigma2_hat)
mse_values  = np.array([np.mean((y - (theta[0] + t1 * X_raw.ravel()))**2)
                         for t1 in theta1_grid])

print(f"OLS optimal slope  θ₁* = {theta[1]:.4f}")
print(f"Max log-likelihood at θ₁ = {theta1_grid[np.argmax(ll_values)]:.4f}")
print(f"Min MSE           at θ₁ = {theta1_grid[np.argmin(mse_values)]:.4f}")
print("All three match — MLE ≡ OLS under Gaussian noise.")

In [ ]:
# ── Side-by-side: Log-likelihood maximisation vs MSE minimisation ─────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: log-likelihood landscape ──────────────────────────────────────────
ax = axes[0]
ax.plot(theta1_grid, ll_values, color='steelblue', linewidth=2)
ax.axvline(theta[1], color='crimson', linestyle='--', linewidth=1.5,
           label=f'MLE optimum θ₁* = {theta[1]:.3f}')
ax.scatter([theta[1]], [ll_values[np.argmax(ll_values)]],
           color='crimson', s=80, zorder=5)
ax.set_xlabel('Slope θ₁')
ax.set_ylabel('Log-Likelihood  L(θ)')
ax.set_title('Probabilistic View:\nMaximise Log-Likelihood', fontsize=12)
ax.legend()

# ── Right: MSE landscape ─────────────────────────────────────────────────────
ax2 = axes[1]
ax2.plot(theta1_grid, mse_values, color='darkorange', linewidth=2)
ax2.axvline(theta[1], color='crimson', linestyle='--', linewidth=1.5,
            label=f'OLS optimum θ₁* = {theta[1]:.3f}')
ax2.scatter([theta[1]], [mse_values[np.argmin(mse_values)]],
            color='crimson', s=80, zorder=5)
ax2.set_xlabel('Slope θ₁')
ax2.set_ylabel('Mean Squared Error')
ax2.set_title('Geometric / OLS View:\nMinimise MSE', fontsize=12)
ax2.legend()

plt.suptitle('MLE ≡ OLS — Same Optimal θ, Different Objectives\n'
             '(Maximising log-likelihood = Minimising MSE under Gaussian noise)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('mle_vs_ols.png', bbox_inches='tight', dpi=110)
plt.show()

## Section 3 — Checking the Gaussian Assumption

The probabilistic interpretation relies on the assumption that errors are Gaussian: ε ~ N(0, σ²).

If this assumption is violated (e.g., errors are right-skewed because house prices have a hard floor at zero), then:
- MSE is no longer the **statistically optimal** loss function
- Our uncertainty estimates (confidence intervals) will be wrong
- We might prefer a different distribution (e.g., log-normal → fit on log(y))

Let's diagnose this with standard residual plots.

In [ ]:
# ── Comprehensive residual diagnostics ───────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# ── (0,0) Residuals vs Fitted — detect heteroscedasticity & non-linearity ───
ax = axes[0, 0]
ax.scatter(y_hat, residuals, alpha=0.4, color='steelblue', s=20)
ax.axhline(0, color='crimson', linestyle='--', linewidth=1.5)
# Smooth trend line to spot systematic patterns
order = np.argsort(y_hat)
smooth_x = y_hat[order]
smooth_y = pd.Series(residuals[order]).rolling(30, center=True, min_periods=5).mean().values
ax.plot(smooth_x, smooth_y, color='orange', linewidth=2, label='Rolling mean')
ax.set_xlabel('Fitted Values ŷ')
ax.set_ylabel('Residuals')
ax.set_title('Residuals vs Fitted')
ax.legend()

# ── (0,1) Q-Q plot — test Gaussian assumption directly ──────────────────────
ax2 = axes[0, 1]
(osm, osr), (slope, intercept, r) = stats.probplot(residuals, dist='norm')
ax2.scatter(osm, osr, alpha=0.5, color='steelblue', s=20)
# Reference line: what perfect normality looks like
ax2.plot(osm, slope * np.array(osm) + intercept, color='crimson', linewidth=2)
ax2.set_xlabel('Theoretical Quantiles (Normal)')
ax2.set_ylabel('Sample Quantiles')
ax2.set_title('Q-Q Plot of Residuals\n(Deviation from line = non-Gaussian)')

# ── (1,0) Histogram with fitted normal ──────────────────────────────────────
ax3 = axes[1, 0]
ax3.hist(residuals, bins=35, color='steelblue', alpha=0.7,
         edgecolor='white', density=True, label='Residuals')
xr   = np.linspace(residuals.min(), residuals.max(), 300)
ax3.plot(xr, stats.norm.pdf(xr, residuals.mean(), residuals.std()),
         color='crimson', linewidth=2.5, label='N(0, σ²)')
ax3.set_xlabel('Residual')
ax3.set_ylabel('Density')
ax3.set_title('Residual Distribution vs Gaussian')
ax3.legend()

# ── (1,1) Scale-Location plot — check for heteroscedasticity ────────────────
ax4 = axes[1, 1]
sqrt_abs_res = np.sqrt(np.abs(residuals))
ax4.scatter(y_hat, sqrt_abs_res, alpha=0.4, color='steelblue', s=20)
ax4.plot(smooth_x,
         pd.Series(sqrt_abs_res[order]).rolling(30, center=True, min_periods=5).mean().values,
         color='orange', linewidth=2, label='Rolling mean')
ax4.set_xlabel('Fitted Values ŷ')
ax4.set_ylabel('√|Residuals|')
ax4.set_title('Scale-Location Plot\n(Flat line = equal variance = homoscedastic)')
ax4.legend()

plt.suptitle('Residual Diagnostics — California Housing (MedInc → HouseVal)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('residual_diagnostics.png', bbox_inches='tight', dpi=110)
plt.show()

# Formal normality test
stat, p_val = stats.shapiro(residuals[:200])   # Shapiro only reliable for n < 5000
print(f"\nShapiro-Wilk test on residuals: stat={stat:.4f}, p={p_val:.4e}")
print(f"If p < 0.05: reject H₀ (normality) → p = {p_val:.4e} → {'REJECT' if p_val < 0.05 else 'FAIL TO REJECT'}")

## Section 4 — Connecting to Regularisation: MAP Estimation

### From MLE to MAP

MLE finds the θ that maximises P(y | X, θ). But what if we have **prior beliefs** about θ?

**MAP (Maximum A Posteriori)** finds θ that maximises:

$$\theta_{\text{MAP}} = \arg\max_\theta \underbrace{P(y \mid X, \theta)}_{\text{likelihood}} \cdot \underbrace{P(\theta)}_{\text{prior}}$$

### Gaussian Prior → Ridge Regression

If we put a Gaussian prior on θ: P(θ) = N(0, (1/λ)I), then:

$$\log P(\theta) = -\frac{\lambda}{2} \|\theta\|^2 + \text{const}$$

Adding this to the log-likelihood, MAP maximisation becomes:

$$\theta_{\text{MAP}} = \arg\min_\theta \sum_i (y_i - x_i^\top\theta)^2 + \lambda \|\theta\|^2$$

This is exactly **Ridge Regression**. The regularisation strength λ encodes our prior belief about how large the weights should be.

### Laplace Prior → Lasso Regression

If we use a Laplace prior P(θ) ∝ exp(−λ|θ|), MAP becomes:

$$\theta_{\text{MAP}} = \arg\min_\theta \sum_i (y_i - x_i^\top\theta)^2 + \lambda \|\theta\|_1$$

This is **Lasso Regression** — and the Laplace prior's sharp peak at zero explains *why* Lasso produces sparse solutions.

In [ ]:
# ── MAP demo: Ridge vs OLS — how the prior pulls θ toward zero ───────────────
# Standardise X so regularisation treats features equally
scaler = StandardScaler()
X_std  = scaler.fit_transform(X_raw)     # standardise MedInc only
X_std_b = np.column_stack([np.ones(n), X_std])  # re-add bias column

# OLS on standardised data (λ = 0, no prior)
theta_ols = np.linalg.inv(X_std_b.T @ X_std_b) @ (X_std_b.T @ y)

# Ridge = MAP with Gaussian prior, analytical solution: (XᵀX + λI)⁻¹ Xᵀy
# Note: we typically don't regularise the intercept (index 0)
lambda_values = [0.01, 0.1, 1.0, 5.0, 20.0, 100.0]
ridge_slopes  = []

for lam in lambda_values:
    # Build regularisation matrix — zero for intercept, λ for slope
    reg_matrix   = lam * np.eye(X_std_b.shape[1])
    reg_matrix[0, 0] = 0                           # don't penalise intercept
    theta_ridge  = np.linalg.inv(X_std_b.T @ X_std_b + reg_matrix) @ (X_std_b.T @ y)
    ridge_slopes.append(theta_ridge[1])            # collect the slope estimate

print("Effect of Gaussian prior (Ridge) on slope estimate:")
print(f"{'λ':>8}  {'θ₁ (MAP/Ridge)':>15}  {'% shrinkage':>12}")
print("-" * 40)
for lam, slope in zip(lambda_values, ridge_slopes):
    shrinkage = (1 - slope / theta_ols[1]) * 100
    print(f"{lam:>8.2f}  {slope:>15.4f}  {shrinkage:>11.1f}%")
print(f"{'OLS (λ=0)':>8}  {theta_ols[1]:>15.4f}  {'(baseline)':>12}")

In [ ]:
# ── Visualise prior distributions: Gaussian vs Laplace ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Left: Prior shapes ───────────────────────────────────────────────────────
ax = axes[0]
theta_range = np.linspace(-3, 3, 400)
ax.plot(theta_range, stats.norm.pdf(theta_range, 0, 1),
        color='steelblue', linewidth=2.5, label='Gaussian prior → Ridge (L2)')
ax.plot(theta_range, stats.laplace.pdf(theta_range, 0, 1),
        color='darkorange', linewidth=2.5, label='Laplace prior  → Lasso (L1)')
ax.axvline(0, color='grey', linestyle=':', linewidth=1)
ax.set_xlabel('θ (weight value)')
ax.set_ylabel('Prior probability P(θ)')
ax.set_title('Prior Distributions\n(Gaussian vs Laplace)', fontsize=12)
ax.legend(fontsize=9)

# ── Middle: Ridge shrinkage path ─────────────────────────────────────────────
ax2 = axes[1]
ax2.semilogx(lambda_values, ridge_slopes, 'o-', color='steelblue',
             linewidth=2, markersize=7)
ax2.axhline(theta_ols[1], color='crimson', linestyle='--', linewidth=1.5,
            label=f'OLS (λ→0): {theta_ols[1]:.3f}')
ax2.set_xlabel('Regularisation λ (log scale)')
ax2.set_ylabel('Slope θ₁ estimate')
ax2.set_title('Ridge Shrinkage Path\n(Larger λ = stronger prior = more shrinkage)', fontsize=12)
ax2.legend()

# ── Right: Summary diagram of MLE / MAP connection ───────────────────────────
ax3 = axes[2]
ax3.axis('off')
summary = [
    (0.50, 0.90, 'Statistical Interpretation of Regression', 'center', 13, 'bold'),
    (0.50, 0.75, 'Prior on θ    →    Loss function', 'center', 11, 'normal'),
    (0.50, 0.65, '─────────────────────────────────', 'center', 10, 'normal'),
    (0.50, 0.55, 'None (flat prior)  →  OLS  (MSE)', 'center', 11, 'normal'),
    (0.50, 0.43, 'Gaussian N(0,λI)  →  Ridge  (L2)', 'center', 11, 'normal'),
    (0.50, 0.31, 'Laplace Lap(0,λ)  →  Lasso  (L1)', 'center', 11, 'normal'),
    (0.50, 0.17, 'Regularisation = encoding prior beliefs', 'center', 10, 'italic'),
    (0.50, 0.07, 'about the size/sparsity of weights', 'center', 10, 'italic'),
]
for x, y_pos, txt, ha, fs, fw in summary:
    ax3.text(x, y_pos, txt, ha=ha, va='center', fontsize=fs,
             fontweight=fw, transform=ax3.transAxes,
             color='black' if fw != 'italic' else 'dimgray')

ax3.set_facecolor('#f5f7fa')

plt.suptitle('MAP Estimation: From MLE to Regularised Regression',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('map_vs_mle.png', bbox_inches='tight', dpi=110)
plt.show()

In [ ]:
# ── Verify MAP/Ridge formula against sklearn Ridge ───────────────────────────
# sklearn Ridge uses the same formula: (XᵀX + λI)⁻¹ Xᵀy

lambda_test = 5.0

# Our manual MAP estimate
reg_matrix  = lambda_test * np.eye(X_std_b.shape[1])
reg_matrix[0, 0] = 0                         # no penalty on intercept
theta_map_manual = np.linalg.inv(X_std_b.T @ X_std_b + reg_matrix) @ (X_std_b.T @ y)

# sklearn Ridge (note: sklearn divides λ by 2n internally in some versions)
# We use fit_intercept=False because we included bias in X_std_b manually
ridge_sk = Ridge(alpha=lambda_test, fit_intercept=False)
ridge_sk.fit(X_std_b, y)

print(f"λ = {lambda_test}")
print(f"  Manual MAP  θ = {theta_map_manual}")
print(f"  sklearn Ridge θ = {ridge_sk.coef_}")
print()
# Note: sklearn's Ridge solves 1/(2n) * RSS + α/2 * ||θ||²
# Our formulation solves RSS + λ * ||θ||² — so effective α = λ/(1)
# They match for alpha = lambda if we are consistent about the 1/n factor
print("NOTE: sklearn Ridge uses a slightly different scaling convention.")
print("Both converge as n→∞ and produce qualitatively identical shrinkage.")

## Section 5 — The Two Interpretations Side by Side

Let's build a final summary visualisation that places both interpretations in one frame.

In [ ]:
# ── Summary: dual interpretation diagram ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Left: Geometric projection diagram (schematic, 2D for clarity) ───────────
ax = axes[0]
ax.set_xlim(-0.5, 4.5)
ax.set_ylim(-0.5, 4.5)

# Draw the "column space" as a horizontal line (the wall)
ax.axhline(y=1.5, xmin=0.05, xmax=0.95, color='steelblue',
           linewidth=4, label='Column space of X (the "wall")', alpha=0.8)
ax.text(2.2, 1.25, 'col(X): all possible Xθ', fontsize=10, color='steelblue')

# The true y (high up in space)
y_true_pt = (2.0, 3.5)
y_hat_pt  = (2.0, 1.5)  # projection straight down (orthogonal)
ax.scatter(*y_true_pt, color='crimson', s=120, zorder=5)
ax.text(y_true_pt[0]+0.1, y_true_pt[1]+0.1, 'y  (true prices\nin ℝⁿ)',
        fontsize=10, color='crimson', va='bottom')

ax.scatter(*y_hat_pt, color='darkorange', s=120, zorder=5)
ax.text(y_hat_pt[0]+0.1, y_hat_pt[1]-0.3, 'ŷ = Xθ  (projection)',
        fontsize=10, color='darkorange')

# Residual arrow (perpendicular to wall)
ax.annotate('', xy=y_hat_pt, xytext=y_true_pt,
            arrowprops=dict(arrowstyle='->', color='green', lw=2.5))
ax.text(2.1, 2.5, 'y − ŷ\n(perpendicular)', fontsize=9.5, color='green')

# Right-angle marker
sq_size = 0.15
ax.plot([y_hat_pt[0], y_hat_pt[0]+sq_size, y_hat_pt[0]+sq_size],
        [y_hat_pt[1]+sq_size, y_hat_pt[1]+sq_size, y_hat_pt[1]],
        color='green', linewidth=1.5)

ax.set_title('Geometric Interpretation\nŷ = Projection of y onto col(X)', fontsize=12)
ax.set_xlabel('Feature space dimension 1')
ax.set_ylabel('Feature space dimension 2')
ax.legend(fontsize=9, loc='upper left')

# ── Right: Probabilistic interpretation ──────────────────────────────────────
ax2 = axes[1]

# Show Gaussian distribution of y given Xθ at three x-values
x_pts    = [1.5, 3.0, 4.5]
y_center = [theta[0] + theta[1]*xp for xp in x_pts]  # predicted means
colors   = ['steelblue', 'darkorange', 'green']

for xp, yc, col in zip(x_pts, y_center, colors):
    y_vals  = np.linspace(yc - 1.5, yc + 1.5, 200)
    density = stats.norm.pdf(y_vals, yc, 0.5)          # σ=0.5 for visual clarity
    # Horizontal density bump at each x-point
    ax2.plot(xp + density * 2.5, y_vals, color=col, linewidth=2)
    ax2.fill_betweenx(y_vals, xp, xp + density * 2.5,
                      color=col, alpha=0.2)
    ax2.axhline(yc, xmin=(xp-0.4)/6, xmax=(xp+1.2)/6,
                color=col, linestyle='--', linewidth=1.2)

# Draw the regression line
xr_diag = np.linspace(0.5, 5.5, 200)
yr_diag = theta[0] + theta[1] * xr_diag
ax2.plot(xr_diag, yr_diag, color='crimson', linewidth=2.5, label='Xθ (predicted mean)')
ax2.set_title('Probabilistic Interpretation\ny | X,θ ~ N(Xθ, σ²)', fontsize=12)
ax2.set_xlabel('Median Income (MedInc)')
ax2.set_ylabel('House Value (predicted mean ± noise)')
ax2.set_xlim(0.5, 7)
ax2.set_ylim(0.5, 5)
ax2.legend(fontsize=9)

plt.suptitle('Two Faces of OLS — Same Formula, Two Deep Insights', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('dual_interpretation.png', bbox_inches='tight', dpi=110)
plt.show()

In [ ]:
# ── Comprehensive summary printout ───────────────────────────────────────────
print("=" * 60)
print("  NOTEBOOK 05 — KEY RESULTS SUMMARY")
print("=" * 60)
print()
print("Dataset: California Housing (MedInc → MedHouseVal, n=300)")
print(f"  OLS intercept θ₀ = {theta[0]:.4f}")
print(f"  OLS slope     θ₁ = {theta[1]:.4f}")
print()
print("Geometric Interpretation:")
print(f"  Residuals orthogonal to X?  {np.allclose(X.T @ residuals, 0, atol=1e-6)}")
print(f"  Residual mean (≈0)?         {residuals.mean():.2e}")
print(f"  RSS = {(residuals**2).sum():.2f}")
print()
print("Probabilistic Interpretation:")
print(f"  MLE optimum at θ₁ =  {theta1_grid[np.argmax(ll_values)]:.4f}  ← same as OLS")
print(f"  MSE optimum at θ₁ =  {theta1_grid[np.argmin(mse_values)]:.4f}  ← same as MLE")
print(f"  Estimated noise σ² =  {sigma2_hat:.4f}")
print()
print("MAP / Regularisation:")
print(f"  Gaussian prior → Ridge (L2 regularisation)")
print(f"  Laplace prior  → Lasso (L1 regularisation)")
print(f"  At λ=5: Ridge shrinks θ₁ from {theta_ols[1]:.4f} to {ridge_slopes[4]:.4f}")
print()
stat_sw, p_sw = stats.shapiro(residuals[:200])
print(f"Gaussian assumption test (Shapiro-Wilk): p = {p_sw:.4e}")
print(f"  → {'Normality REJECTED' if p_sw < 0.05 else 'Cannot reject normality'}")
print("  → Tails are heavier than Gaussian due to house price ceiling at $500k")
print("=" * 60)

## Self-Check Questions

Test your understanding. Try to answer without looking, then expand.

---

**Question 1:** The geometric interpretation says residuals are orthogonal to the column space of X. What does "orthogonal" mean geometrically, and why does this make OLS the "best" solution?

<details>
<summary>▶ Click to reveal answer</summary>

**Orthogonal** means perpendicular — the residual vector (y − ŷ) meets the column space of X at a 90° angle. This is the *only* way to achieve the minimum distance.

To see why: imagine dropping a perpendicular from a point to a line. The perpendicular path is the *shortest* path from the point to the line. Any other path is longer (by Pythagoras). In higher dimensions, the same principle applies: the residuals point in the direction *not captured by X*, and they are as short as possible precisely because they are perpendicular.

Mathematically: if the residuals were not orthogonal to some column xⱼ, then the inner product xⱼᵀ(y − ŷ) ≠ 0, and we could improve ŷ by moving it a little in the xⱼ direction — contradicting the assumption that it was optimal. So at the optimum, all inner products must be zero: **Xᵀ(y − ŷ) = 0** — the normal equations.

</details>

---

**Question 2:** If the errors in your house price model are NOT normally distributed (e.g., right-skewed), what does that imply about using MSE as your loss function?

<details>
<summary>▶ Click to reveal answer</summary>

The probabilistic justification for MSE is that it is the MLE loss *under Gaussian noise*. If errors are right-skewed (as often happens with house prices, since they can't go below zero but can be very large):

1. **MSE is no longer statistically optimal** — a different likelihood (e.g., log-normal, Gamma) would give a better-motivated loss function.
2. **MSE overpenalises large errors** (because it squares them), making the model chase outliers.
3. **Confidence intervals** derived from the Gaussian assumption will be wrong.
4. **Better alternatives**: fit on log(y) and model log-prices as Gaussian; or use Mean Absolute Error (MAE) which corresponds to a Laplace likelihood and is more robust to outliers.

Practical sign: the Q-Q plot will show the residuals curling away from the reference line at the right tail, and the residuals-vs-fitted plot will show increasing variance for larger predictions (heteroscedasticity).

</details>

---

**Question 3:** Ridge regression adds a prior P(θ) = N(0, λI) on the weights. How does this connect MLE to MAP?

<details>
<summary>▶ Click to reveal answer</summary>

- **MLE** maximises the likelihood P(y | X, θ) alone, ignoring any prior knowledge about θ. Result: OLS with no regularisation.
- **MAP** maximises the posterior P(θ | X, y) ∝ P(y | X, θ) · P(θ). It combines the likelihood with a prior.

When the prior is Gaussian: P(θ) = N(0, (1/λ)I):

  log P(θ) = −(λ/2) ||θ||²  + const

Adding this to the log-likelihood and negating gives:

  MAP objective = MSE + λ||θ||²

This is Ridge Regression. The Gaussian prior encodes the belief that weights should be *small*. Larger λ = stronger belief = more shrinkage toward zero. As λ → 0, MAP → MLE → OLS. As λ → ∞, all weights → 0 (we only trust the prior, not the data).

</details>

In [ ]:
# ── Bonus: visualise the Gaussian noise model at a specific income level ──────
# Fix income at median, show the Gaussian predictive distribution

income_val = df_sample['MedInc'].median()           # fix at median income
y_pred_at_median = theta[0] + theta[1] * income_val # predicted mean
sigma_hat = np.sqrt(sigma2_hat)                      # estimated noise std

fig, ax = plt.subplots(figsize=(9, 5))

y_range = np.linspace(y_pred_at_median - 3.5*sigma_hat,
                       y_pred_at_median + 3.5*sigma_hat, 400)
density = stats.norm.pdf(y_range, y_pred_at_median, sigma_hat)

ax.plot(y_range, density, color='steelblue', linewidth=2.5,
        label=f'P(y | MedInc={income_val:.2f}, θ) = N({y_pred_at_median:.2f}, {sigma_hat:.2f}²)')
ax.fill_between(y_range, density, alpha=0.25, color='steelblue')

# Mark the predicted mean
ax.axvline(y_pred_at_median, color='crimson', linewidth=2, linestyle='--',
           label=f'Predicted mean ŷ = {y_pred_at_median:.3f} ($100k)')

# Mark ±1σ and ±2σ
for k, col in zip([1, 2], ['orange', 'green']):
    ax.axvline(y_pred_at_median + k*sigma_hat, color=col, linewidth=1.2,
               linestyle=':', alpha=0.8, label=f'+{k}σ / -{k}σ')
    ax.axvline(y_pred_at_median - k*sigma_hat, color=col, linewidth=1.2,
               linestyle=':', alpha=0.8)

ax.set_xlabel('Predicted House Value ($100k)')
ax.set_ylabel('Probability Density')
ax.set_title(f'Probabilistic Model: P(y | MedInc = {income_val:.2f}, θ)\n'
             'Gaussian uncertainty around the predicted mean', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('gaussian_predictive.png', bbox_inches='tight', dpi=110)
plt.show()

print(f"At MedInc = {income_val:.2f}:")
print(f"  Predicted house value  = ${y_pred_at_median*100:.0f}k")
print(f"  Uncertainty (±1σ)      = ±${sigma_hat*100:.0f}k")
print(f"  95% interval           = [${(y_pred_at_median-2*sigma_hat)*100:.0f}k,  ${(y_pred_at_median+2*sigma_hat)*100:.0f}k]")